In [1]:
# Explanation: This cell generates all homomorphic images of the 4-uniform tight cycle C_13 on at most 6 vertices, then excludes those images from the theory.

from collections import defaultdict
from itertools import permutations

target_size = 6
cycle_length = 13
uniformity = 4


def tight_cycle_edges(n, r=4):
    return [tuple((i + j) % n for j in range(r)) for i in range(n)]


def canonical_edge_set(num_vertices, edges):
    """Return a canonical representative of an unlabeled 4-graph."""
    best = None
    for perm in permutations(range(num_vertices)):
        relabeled = tuple(sorted({tuple(sorted(perm[v] for v in edge)) for edge in edges}))
        if best is None or relabeled < best:
            best = relabeled
    return best


def generate_tight_cycle_homomorphic_images(n, max_vertices, r=4):
    """Generate all homomorphic images of C_n^(r) with at most max_vertices vertices, up to isomorphism."""
    cycle_edges = tight_cycle_edges(n, r)
    images = defaultdict(dict)
    assignment = [None] * n
    assignment[0] = 0

    def complete_edge_is_valid(edge):
        colors = [assignment[v] for v in edge]
        if any(color is None for color in colors):
            return True
        return len(set(colors)) == r

    def partial_assignment_is_valid():
        return all(complete_edge_is_valid(edge) for edge in cycle_edges)

    def record_image(current_max):
        num_vertices = current_max + 1
        edges = tuple(sorted({tuple(sorted(assignment[v] for v in edge)) for edge in cycle_edges}))
        key = canonical_edge_set(num_vertices, edges)
        images[num_vertices].setdefault(key, [list(edge) for edge in key])

    def backtrack(pos, current_max):
        if pos == n:
            if partial_assignment_is_valid():
                record_image(current_max)
            return

        upper = min(current_max + 1, max_vertices - 1)
        for color in range(upper + 1):
            assignment[pos] = color
            if partial_assignment_is_valid():
                backtrack(pos + 1, max(current_max, color))
            assignment[pos] = None

    backtrack(1, 0)
    return images


C13_images_by_size = generate_tight_cycle_homomorphic_images(cycle_length, target_size, uniformity)
C13_homomorphic_images = [
    (num_vertices, edges)
    for num_vertices in sorted(C13_images_by_size)
    for edges in sorted(C13_images_by_size[num_vertices].values())
]
C13_image_theories = [
    FourGraphTheory.p(num_vertices, edges=edges)
    for num_vertices, edges in C13_homomorphic_images
]

edge = FourGraphTheory(4, edges=[[0, 1, 2, 3]])
FourGraphTheory.exclude(C13_image_theories)
FGT = FourGraphTheory

print("Homomorphic images of 4-uniform C_13 with at most 6 vertices:")
for num_vertices in range(1, target_size + 1):
    print(f"and size {num_vertices}: ", len(C13_images_by_size.get(num_vertices, {})))
print("total: ", len(C13_homomorphic_images))

print("Excluded homomorphic images:")
name_counts = defaultdict(int)
for num_vertices, edges in C13_homomorphic_images:
    image_index = name_counts[num_vertices]
    name_counts[num_vertices] += 1
    print(f"C13_J{num_vertices}_{image_index} = FourGraphTheory.p({num_vertices}, edges={edges})")

print("Number of structures without these C_13 homomorphic images")
print("and size 5: ", len(FGT.generate(5)))
print("and size 6: ", len(FGT.generate(6)))
# print("and size 7: ", len(FGT.generate(7)))


Homomorphic images of 4-uniform C_13 with at most 6 vertices:
and size 1:  0
and size 2:  0
and size 3:  0
and size 4:  0
and size 5:  1
and size 6:  16
total:  17
Excluded homomorphic images:
C13_J5_0 = FourGraphTheory.p(5, edges=[[0, 1, 2, 3], [0, 1, 2, 4], [0, 1, 3, 4], [0, 2, 3, 4], [1, 2, 3, 4]])
C13_J6_0 = FourGraphTheory.p(6, edges=[[0, 1, 2, 3], [0, 1, 2, 4], [0, 1, 2, 5], [0, 1, 3, 4], [0, 1, 3, 5], [0, 2, 3, 4], [0, 2, 4, 5], [0, 3, 4, 5], [1, 2, 3, 4]])
C13_J6_1 = FourGraphTheory.p(6, edges=[[0, 1, 2, 3], [0, 1, 2, 4], [0, 1, 2, 5], [0, 1, 3, 4], [0, 1, 3, 5], [0, 2, 3, 4], [0, 2, 4, 5], [0, 3, 4, 5], [1, 2, 3, 5]])
C13_J6_2 = FourGraphTheory.p(6, edges=[[0, 1, 2, 3], [0, 1, 2, 4], [0, 1, 2, 5], [0, 1, 3, 4], [0, 1, 3, 5], [0, 2, 3, 4], [1, 2, 3, 4]])
C13_J6_3 = FourGraphTheory.p(6, edges=[[0, 1, 2, 3], [0, 1, 2, 4], [0, 1, 2, 5], [0, 1, 3, 4], [0, 1, 3, 5], [0, 2, 3, 4], [1, 2, 3, 5]])
C13_J6_4 = FourGraphTheory.p(6, edges=[[0, 1, 2, 3], [0, 1, 2, 4], [0, 1, 2, 5], [0, 1, 3

In [2]:
# Explanation: This cell runs the exact flag algebra SDP for the edge density upper bound using the balanced bipartite 0001/0111 construction as the rounding guide.

trip_constr = FGT.blowup_construction(
    target_size,
    ["X0", "X1"],
    edges=[[0, 0, 0, 1], [0, 1, 1, 1]],
    symbolic=True
)
eval_constr = trip_constr.subs([1/2, 1/2])

FGT.optimize(
    edge,
    target_size,
    exact=True,
    file="FourGraph_C13_6vtx",
    construction=eval_constr,
    denom=1024*15*3*5,
    slack_threshold=2e-6
)


Base flags generated, their number is 69
The relevant ftypes are constructed, their number is 3
Block sizes before symmetric/asymmetric change is applied: [2, 16, 15]


Done with mult table for Ftype on 4 points with edges=(0123): : 3it [00:01,  1.77it/s]


Tables finished
Constraints finished
Adjusting table with kernels from construction
Running SDP after kernel correction. Used block sizes are [1, 3, 9, 3, 8, -69, -2]
CSDP 6.2.0
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 1.00e+00 Pobj: -1.7546746e+01 Ad: 7.55e-01 Dobj:  5.5432939e-01 
Iter:  2 Ap: 1.00e+00 Pobj: -1.8099493e+01 Ad: 9.52e-01 Dobj: -3.1097676e-01 
Iter:  3 Ap: 1.00e+00 Pobj: -1.6539645e+01 Ad: 9.32e-01 Dobj: -3.5787456e-01 
Iter:  4 Ap: 1.00e+00 Pobj: -6.6843359e+00 Ad: 9.49e-01 Dobj: -3.6064738e-01 
Iter:  5 Ap: 1.00e+00 Pobj: -9.3080149e-01 Ad: 8.65e-01 Dobj: -3.6377712e-01 
Iter:  6 Ap: 1.00e+00 Pobj: -6.4841836e-01 Ad: 6.72e-01 Dobj: -3.9371057e-01 
Iter:  7 Ap: 1.00e+00 Pobj: -5.8926116e-01 Ad: 6.91e-01 Dobj: -4.3991943e-01 
Iter:  8 Ap: 9.20e-01 Pobj: -5.4593016e-01 Ad: 7.31e-01 Dobj: -4.6440533e-01 
Iter:  9 Ap: 1.00e+00 Pobj: -5.2450069e-01 Ad: 7.25e-01 Dobj: -4.7813916e-01 
Iter: 10 Ap: 1.00e+00 Pobj: -5.0756836e-01

 33%|████████████████████████▎                                                | 1/3 [00:00<00:00, 28.28it/s]


Rounded X matrix ((5, Ftype on 4 points with edges=(), 6), 1) is not semidefinite: -0.012109906217762527
Rounding based on construction was unsuccessful
Rounding X matrices


100%|████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 674.95it/s]


Calculating resulting bound


100%|█████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 42.83it/s]

The rounded result is -13879/27648
Final rounded bound is 13879/27648


13879/27648